# Pandas II — Limpieza de datos y construcción de una tabla analítica

## Objetivos
- Diagnosticar calidad de datos con rapidez.
- Limpiar duplicados, nulos y texto.
- Validar tipos y reglas de negocio básicas.
- Unir tablas con `merge` y construir una `fact_sales`.
- Dejar el dataset listo para análisis en Pandas III.

---
## 0. Setup

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

np.random.seed(42)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)


---
## 1. Datos de trabajo

Trabajaremos con tres tablas con problemas realistas:
- `customers`: dimensión de clientes
- `products`: dimensión de productos
- `transactions`: tabla de transacciones

La idea es limpiarlas y convertirlas en una tabla analítica única.

In [ ]:
# --- Customers ---
#Paso 1: Crear los datos para el dataframe de customers
n_customers = 120
customer_ids = np.arange(1000, 1000 + n_customers)

cities = np.random.choice(['Madrid', 'Barcelona', 'Valencia', 'Sevilla'], size=n_customers)
segments = np.random.choice(['Standard', 'Premium'], size=n_customers, p=[0.75, 0.25])
ages = np.random.normal(39, 12, size=n_customers).round().astype(float)
names = [f' customer_{i} ' for i in range(n_customers)]

#Paso 2: Crear el dataset Customers
customers = pd.DataFrame({
    'customer_id': customer_ids,
    'name': names,
    'city': cities,
    'segment': segments,
    'age': ages,
})
#Paso 3: ensuciar los datos del dataframe Customers
customers.loc[np.random.choice(customers.index, 10, replace=False), 'age'] = np.nan
customers.loc[np.random.choice(customers.index, 6, replace=False), 'city'] = None
customers = pd.concat([customers, customers.sample(8, random_state=42)], ignore_index=True)

# --- Products ---
#Paso 1: Crear los datos para el dataframe de Productos
product_ids = np.arange(200, 240)
categories = np.random.choice(['Electronics', 'Home', 'Sports', 'Books'], size=len(product_ids))
base_prices = np.random.uniform(5, 250, size=len(product_ids)).round(2)
product_names = [f' Product_{pid} ' for pid in product_ids]

#Paso 2: Crear el data Frame de productos
products = pd.DataFrame({
    'product_id': product_ids,
    'product_name': product_names,
    'category': categories,
    'base_price': base_prices,
})
#Paso 3: ensuciar los datos del dataframe Productos
products.loc[np.random.choice(products.index, 5, replace=False), 'category'] = ' electronics '
products.loc[np.random.choice(products.index, 4, replace=False), 'base_price'] = np.nan
products = pd.concat([products, products.sample(5, random_state=1)], ignore_index=True)

# --- Transactions(Pedidos- un consumidor hace pedido de productos) ---
#Paso 1: Crear los datos para dataframe Trasacciones
n_tx = 900
tx_ids = np.arange(5000, 5000 + n_tx)

tx_customer = np.random.choice(customers['customer_id'].unique(), size=n_tx)
tx_product = np.random.choice(products['product_id'].unique(), size=n_tx)
quantity = np.random.randint(1, 6, size=n_tx)
discount = np.random.choice([0.0, 0.05, 0.10, np.nan], size=n_tx, p=[0.55, 0.2, 0.15, 0.10])

dates = pd.to_datetime('2025-01-01') + pd.to_timedelta(np.random.randint(0, 90, size=n_tx), unit='D')
#Paso 2: Crear el data set de transacciones
transactions = pd.DataFrame({
    'tx_id': tx_ids,
    'customer_id': tx_customer,
    'product_id': tx_product,
    'quantity': quantity,
    'discount': discount,
    'tx_date': dates,
})

#Paso 3: Ensuciar los datos del dataset transacciones
transactions.loc[np.random.choice(transactions.index, 7, replace=False), 'product_id'] = np.nan
transactions = pd.concat([transactions, transactions.sample(12, random_state=7)], ignore_index=True)

print('customers', customers.shape)
display(customers.head())
print('products', products.shape)
display( products.head())
print('transactions', transactions.shape)
display( transactions.head())


customers (128, 5)


,customer_id,name,city,segment,age
0,1000,customer_0,Barcelona,Standard,NaN
1,1001,customer_1,Barcelona,Standard,56.0
2,1002,customer_2,Sevilla,Standard,31.0
3,1003,customer_3,Valencia,Standard,7.0
4,1004,customer_4,Madrid,Standard,NaN


products (45, 4)


,product_id,product_name,category,base_price
0,200,Product_200,Sports,239.46
1,201,Product_201,electronics,153.08
2,202,Product_202,electronics,213.16
3,203,Product_203,Electronics,NaN
4,204,Product_204,Sports,141.66


transactions (900, 6)


,tx_id,customer_id,product_id,quantity,discount,tx_date
0,5000,1024,224,1,0.05,2025-03-21
1,5001,1001,216,3,NaN,2025-03-21
2,5002,1013,219,3,0.00,2025-01-23
3,5003,1016,227,2,0.00,2025-01-16
4,5004,1104,234,2,0.00,2025-03-17


---
## 2. Diagnóstico inicial

Antes de tocar nada, conviene mirar:
- tamaño
- tipos
- nulos
- duplicados
- claves potencialmente problemáticas

### Ejemplo: forma, nulos y duplicados

In [ ]:
print(customers.shape)
print(products.shape)
print(transactions.shape)

display(customers.isna())#muestra la tabla y en cada celda true o false si el valor es null
display(customers.isna().sum())
display(products.isna().sum())
display(transactions.isna().sum())

print('Duplicated rows customers:', customers.duplicated().sum())
print('Duplicated rows products:', products.duplicated().sum())
print('Duplicated rows transactions:', transactions.duplicated().sum())

(128, 5)
(45, 4)
(900, 6)


,customer_id,name,city,segment,age
0,False,False,False,False,True
1,False,False,False,False,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,True
...,...,...,...,...,...
123,False,False,False,False,False
124,False,False,False,False,False
125,False,False,False,False,False
126,False,False,False,False,False


customer_id     0
name            0
city            6
segment         0
age            11
dtype: int64

product_id      0
product_name    0
category        0
base_price      5
dtype: int64

tx_id            0
customer_id      0
product_id       0
quantity         0
discount       102
tx_date          0
dtype: int64

Duplicated rows customers: 8
Duplicated rows products: 5
Duplicated rows transactions: 0


### Ejercicio 1
1. Ejecuta `info()` para las tres tablas.
2. Calcula duplicados completos en cada una.
3. Localiza IDs duplicados en `customer_id`, `product_id` y `tx_id`.

In [ ]:
# TODO
#customers.info()
#products.info()
#transactions.info()
#Calcular duplicados
print("Columnas duplicadas en customers: ",customers.duplicated().sum())
print("Columnas duplicadas en products: ",products.duplicated().sum())
print("Columnas duplicadas en transactions: ",transactions.duplicated().sum())

#Localizar
display(customers[customers['customer_id'].duplicated(keep=False)].sort_values('customer_id'))
# primero ordenar el dataset, luego filtrar por los valores que estan duplicados, 
# el keep=false es para que nos muestre todas las instancias duplicadas
display(products[products['product_id'].duplicated(keep=False)].sort_values('product_id'))
display(transactions[transactions['tx_id'].duplicated(keep=False)].sort_values('tx_id'))

Columnas duplicadas en customers:  8
Columnas duplicadas en products:  5
Columnas duplicadas en transactions:  0


,customer_id,name,city,segment,age
4,1004,customer_4,Madrid,Standard,NaN
122,1004,customer_4,Madrid,Standard,NaN
127,1010,customer_10,Barcelona,Standard,35.0
10,1010,customer_10,Barcelona,Standard,35.0
26,1026,customer_26,Valencia,Premium,34.0
124,1026,customer_26,Valencia,Premium,34.0
120,1044,customer_44,Madrid,Standard,28.0
44,1044,customer_44,Madrid,Standard,28.0
121,1047,customer_47,Madrid,Standard,42.0
47,1047,customer_47,Madrid,Standard,42.0


,product_id,product_name,category,base_price
2,202,Product_202,electronics,213.16
40,202,Product_202,electronics,213.16
42,203,Product_203,Electronics,NaN
3,203,Product_203,Electronics,NaN
21,221,Product_221,Home,153.58
43,221,Product_221,Home,153.58
44,227,Product_227,Electronics,95.80
27,227,Product_227,Electronics,95.80
41,231,Product_231,Books,165.05
31,231,Product_231,Books,165.05


,tx_id,customer_id,product_id,quantity,discount,tx_date


---
## 3. Duplicados

Hay dos casos muy típicos:
- duplicados exactos
- duplicados por clave

Normalmente primero eliminamos duplicados completos y luego resolvemos la unicidad de las claves.

### Ejemplo: deduplicar por filas y por clave

In [30]:
transactions_dedup = transactions.drop_duplicates()
customers_dedup = customers.drop_duplicates(subset=['customer_id'], keep='first')
products_dedup = products.drop_duplicates(subset=['product_id'], keep='first')

print('transactions:', transactions.shape, '->', transactions_dedup.shape)
print('customers:', customers.shape, '->', customers_dedup.shape)
print('products:', products.shape, '->', products_dedup.shape)

transactions: (900, 6) -> (900, 6)
customers: (128, 5) -> (120, 5)
products: (45, 4) -> (40, 4)


### Ejercicio 2
1. Verifica que `customer_id` y `product_id` quedan únicos tras deduplicar.
2. Comprueba que `tx_id` queda único en `transactions_dedup`.
3. Muestra las filas conflictivas antes de limpiar para justificar la decisión.

In [31]:
# TODO

display(customers_dedup[customers_dedup['customer_id'].duplicated(keep=False)].sort_values('customer_id'))
display(products_dedup[products_dedup['product_id'].duplicated(keep=False)].sort_values('product_id'))
display(transactions_dedup[transactions_dedup['tx_id'].duplicated(keep=False)].sort_values('tx_id'))

,customer_id,name,city,segment,age


,product_id,product_name,category,base_price


,tx_id,customer_id,product_id,quantity,discount,tx_date


---
## 4. Nulos

No todos los nulos se resuelven igual:
- a veces imputamos
- a veces eliminamos
- a veces comparamos varias estrategias

### Ejemplo: limpieza básica de `customers`

In [ ]:
customers_clean = customers_dedup.copy()
customers_clean['age'] = customers_clean['age'].fillna(customers_clean['age'].median())#median es la edad que mas repite
customers_clean['city'] = customers_clean['city'].fillna('Unknown')

display(customers_clean.isna().sum())

### Ejercicio 3
Construye `transactions_clean` a partir de `transactions_dedup`:
1. Rellena `discount` con `0.0`.
2. Elimina filas con `product_id` nulo.
3. Comprueba cuántas filas pierdes.

In [45]:
# TODO
transactions_clean = transactions_dedup.copy()
transactions_clean['age'] = transactions_clean['discount'].fillna(0.0)#median es la edad que mas repite

transactions_clean = transactions_clean.dropna(subset=['product_id'])
print("Antes: ",transactions_dedup.shape[0])
print("Despues: ",transactions_clean.shape[0])


Antes:  900
Despues:  900


### Ejemplo: imputación global frente a imputación por grupo
- Imputar significa rellenar valores faltantes (NaN) con algún valor calculado.
- La diferencia entre imputación global e imputación por grupo es cómo calculas el valor que reemplaza los NaN.

1. Imputación global: 
- Se usa un único valor para toda la columna.
- Por ejemplo, rellenar todos los NaN con la media global:
    df["edad"] = df["edad"].fillna(df["edad"].mean())

2. Imputación por grupo:
- El valor depende del grupo al que pertenece cada fila.
- Por ejemplo:
    - media por ciudad
    - media por género
    - media por categoría

In [47]:
customers_clean_global = customers_dedup.copy()
customers_clean_global['age'] = customers_clean_global['age'].fillna(customers_clean_global['age'].median())

customers_clean_group = customers_dedup.copy()
segment_median_age = customers_clean_group.groupby('segment')['age'].transform('median')
customers_clean_group['age'] = customers_clean_group['age'].fillna(segment_median_age)

print('Global median age:', customers_clean_global['age'].median())
print('Group median age by segment:')
display(customers_clean_group.groupby('segment')['age'].median())

Global median age: 36.0
Group median age by segment:


segment
Premium     37.0
Standard    35.0
Name: age, dtype: float64

### Ejercicio 4
1. Compara nulos restantes y mediana final entre `customers_clean_global` y `customers_clean_group`.
2. Elige una estrategia y guárdala en `customers_clean`.
3. Explica brevemente cuál te parece más defendible.

In [54]:
# TODO
print('Global nulls age:', customers_clean_global['age'].isna().sum())
print('Group nulls age :', customers_clean_group['age'].isna().sum())

print('Global median age:', customers_clean_global['age'].median())
print('Chosen strategy  : group median by segment')

display(customers_clean_group.groupby('segment')['age'].median())

customers_clean = customers_clean_group.copy()
customers_clean['city'] = customers_clean['city'].fillna('Unknown')

display(customers_clean.isna().sum())

Global nulls age: 0
Group nulls age : 0
Global median age: 36.0
Chosen strategy  : group median by segment


segment
Premium     37.0
Standard    35.0
Name: age, dtype: float64

customer_id    0
name           0
city           0
segment        0
age            0
dtype: int64

---
## 5. Limpieza de texto con `.str`

Operaciones muy frecuentes:
- `strip()`
- `lower()` / `title()`
- reemplazos
- control de dominios

### Ejemplo: normalizar categorías y nombres de producto

In [51]:
products_clean = products_dedup.copy()
products_clean['category'] = products_clean['category'].astype(str).str.strip().str.title()
products_clean['product_name'] = products_clean['product_name'].astype(str).str.strip().str.title()

print('Category values after cleaning:')
display(products_clean['category'].value_counts())

Category values after cleaning:


category
Electronics    14
Home           10
Sports          8
Books           8
Name: count, dtype: int64

### Ejercicio 5
1. Limpia `customers_clean['name']` con `.str`.
2. Asegura que `products_clean['category']` solo contiene `{Electronics, Home, Sports, Books}`.
3. Si aparece un valor fuera de dominio, conviértelo a `Unknown`.

In [55]:
# TODO
transactions_test = transactions.copy()

transactions_test['discount_band'] = transactions_test['discount'].apply(lambda d: discount_band(d) if pd.notna(d) else 'N/A')

display(transactions_test['discount_band'].value_counts())

NameError: name 'discount_band' is not defined

---
## 6. `apply()` cuando aporta valor

`apply` es útil cuando la lógica no es trivial o cuando una regla no sale limpia con una sola operación vectorizada.

### Ejemplo: bandas de descuento

In [68]:
def discount_band(d):
    if d == 0:
        return 'none'
    if d < 0.10:
        return 'low'
    return 'high'

transactions_clean = transactions_dedup.copy()
transactions_clean['discount'] = transactions_clean['discount'].fillna(0.0)
transactions_clean = transactions_clean.dropna(subset=['product_id']).copy()
transactions_clean['discount_band'] = transactions_clean['discount'].apply(discount_band)

display(transactions_clean['discount_band'].value_counts())

discount_band
none    613
low     151
high    136
Name: count, dtype: int64

### Ejercicio 6
1. Crea una función `customer_label(row)` con `axis=1`:
   - `VIP_Senior` si `age >= 50` y `segment == 'Premium'`
   - `Premium` si `segment == 'Premium'`
   - `Standard` en otro caso
2. Añade la columna `customer_label` a `customers_clean`.

In [62]:
# TODO
def customer_label(row):
    if row['segment'] == 'Premium':
        if row['age'] >= 50:
            return "VIP_Senior"
        else:
            return "Premium"
    else:
        return "Standar"
    
customers_clean1 = customers_clean_group.copy()
customers_clean1['customer_label'] = customers_clean1.apply(customer_label, axis=1)
print(customers_clean1["customer_label"].value_counts())
display(customers_clean1)
    



    

customer_label
Standar       94
Premium       23
VIP_Senior     3
Name: count, dtype: int64


,customer_id,name,city,segment,age,customer_label
0,1000,customer_0,Barcelona,Standard,35.0,Standar
1,1001,customer_1,Barcelona,Standard,56.0,Standar
2,1002,customer_2,Sevilla,Standard,31.0,Standar
3,1003,customer_3,Valencia,Standard,7.0,Standar
4,1004,customer_4,Madrid,Standard,35.0,Standar
...,...,...,...,...,...,...
115,1115,customer_115,Sevilla,Standard,23.0,Standar
116,1116,customer_116,Barcelona,Standard,65.0,Standar
117,1117,customer_117,Valencia,Premium,30.0,Premium
118,1118,customer_118,Sevilla,Standard,29.0,Standar


---
## 7. Tipos y validaciones

Antes de unir tablas conviene dejar claras algunas reglas mínimas:
- IDs sin nulos
- cantidades válidas
- descuentos en rango
- precios positivos

### Ejemplo: asserts mínimos

In [69]:
assert customers_clean['customer_id'].isna().sum() == 0
assert products_clean['product_id'].isna().sum() == 0
assert (products_clean['base_price'].dropna() > 0).all()
assert (transactions_clean['quantity'] >= 1).all()
#assert transactions_clean['discount'].between(0, 1).all()

print('Basic validation passed')

Basic validation passed


### Ejercicio 7
1. Convierte `customer_id` y `product_id` a enteros donde corresponda.
2. Imputa `base_price` con la mediana de la categoría.
3. Añade al menos 3 `assert` útiles sobre las tablas limpias.

In [70]:
customers_clean['customer_id'] = customers_clean['customer_id'].astype(int)
customers_clean['age'] = customers_clean['age'].astype(int)
products_clean['product_id'] = products_clean['product_id'].astype(int)
transactions_clean['customer_id'] = transactions_clean['customer_id'].astype(int)
transactions_clean['product_id'] = transactions_clean['product_id'].astype(int)
products_clean['base_price'] = products_clean.groupby('category')['base_price'].transform(
lambda s: s.fillna(s.median()))

assert customers_clean['customer_id'].isna().sum() == 0
assert products_clean['product_id'].isna().sum() == 0
assert (products_clean['base_price'] > 0).all()
assert (transactions_clean['quantity'] >= 1).all()
assert transactions_clean['discount'].between(0, 0.1).all()

print('Extended validation passed')

Extended validation passed


---
## 8. `merge` y construcción de `fact_sales`

El objetivo final de esta sesión es construir una tabla analítica a nivel transacción.

### Ejemplo: `left` vs `inner`

In [ ]:
demo_left = pd.DataFrame({'id': [1, 2, 3], 'x': [10, 20, 30]})
demo_right = pd.DataFrame({'id': [2, 3, 4], 'y': [200, 300, 400]})

left_join = demo_left.merge(demo_right, on='id', how='left')
inner_join = demo_left.merge(demo_right, on='id', how='inner')

display(left_join)
display(inner_join)

### Ejercicio 8
1. Construye `right_join` y `outer_join` con las tablas demo.
2. Indica cuál conserva más filas.
3. Explica cuándo usarías `left` en un proceso analítico.

In [ ]:
# TODO
pass

### Ejercicio 9
Construye `fact_sales`:
1. Une `transactions_clean` con `products_clean` por `product_id` con `how='left'`.
2. Une el resultado con `customers_clean` por `customer_id` con `how='left'`.
3. Crea `amount = quantity * base_price * (1 - discount)`.
4. Devuelve un subconjunto de columnas clave para revisar el resultado.

In [ ]:
# TODO
pass

---
## 9. Features mínimas y control final

Una vez creada la fact table, conviene dejar algunas variables analíticas básicas listas para la siguiente sesión.

### Ejemplo: mes y día de la semana

In [ ]:
# Este bloque asume que ya existe fact_sales
# fact_sales['month'] = fact_sales['tx_date'].dt.month
# fact_sales['weekday'] = fact_sales['tx_date'].dt.day_name()

### Ejercicio 10
1. Añade a `fact_sales` las columnas `month` y `weekday`.
2. Crea `age_group` con `pd.cut` (`Young`, `Adult`, `Senior`).
3. Construye una tabla `data_quality` con filas, columnas y nulos totales de `customers_clean`, `products_clean`, `transactions_clean` y `fact_sales`.

In [ ]:
# TODO
pass